# Fine-tune PaddleOCR recognition on IAM Handwriting

Run these cells top to bottom on a **T4 GPU runtime** (Runtime -> Change runtime type -> T4 GPU).

Before running: register at https://fki.tic.heia-fr.ch/databases/iam-handwriting-database and download `ascii/lines.txt` and `data/lines.tgz`. Upload them to Google Drive so you don't have to re-upload each session.

## 1. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THESE to match where you put the IAM files in Drive
LINES_TXT = "/content/drive/MyDrive/iam/lines.txt"
LINES_TGZ = "/content/drive/MyDrive/iam/lines.tgz"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/iam/paddleocr_finetune_output"

import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)


## 2. Clone PaddleOCR and install dependencies

In [ ]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -q -r requirements.txt
!pip install -q paddlepaddle-gpu


## 3. Extract IAM line images

In [ ]:
!mkdir -p /content/iam_lines
!tar -xzf "$LINES_TGZ" -C /content/iam_lines
!echo "Sample of extracted structure:" && find /content/iam_lines -maxdepth 3 | head -20


## 4. Convert IAM labels to PaddleOCR format

Upload `prepare_iam_data.py` (from this repo's `ocr_training/` folder) into the Colab file browser first, or paste its contents into the cell below.

In [ ]:
!python /content/prepare_iam_data.py \
    --lines-txt "$LINES_TXT" \
    --img-root /content/iam_lines \
    --out-dir /content/iam_labels \
    --val-split 0.1

!wc -l /content/iam_labels/train_list.txt /content/iam_labels/val_list.txt
!head -3 /content/iam_labels/train_list.txt


## 5. Download the pretrained PP-OCRv4 English recognition model

This is your fine-tuning starting point -- the same model your local CS-PADDLE-OCR.py script already uses, so you're specializing it rather than training from scratch.

In [ ]:
%cd /content/PaddleOCR
!wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_train.tar
!tar -xf en_PP-OCRv4_rec_train.tar
!ls en_PP-OCRv4_rec_train


## 6. Write the fine-tuning config

In [ ]:
config_yaml = '''
Global:
  use_gpu: true
  epoch_num: 30
  log_smooth_window: 20
  print_batch_step: 50
  save_model_dir: /content/output/iam_finetune
  save_epoch_step: 5
  eval_batch_step: [0, 500]
  cal_metric_during_train: true
  pretrained_model: /content/PaddleOCR/en_PP-OCRv4_rec_train/best_accuracy
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  character_dict_path: ppocr/utils/en_dict.txt
  max_text_length: &max_text_length 100
  infer_mode: false
  use_space_char: true
  distributed: false

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: PPLCNetV3
    scale: 0.95
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 120
            depth: 2
            hidden_dims: 120
            kernel_size: [1, 3]
            use_guide: True
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: *max_text_length

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /
    label_file_list: ["/content/iam_labels/train_list.txt"]
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecConAug:
          prob: 0.5
          image_shape: [48, 320, 3]
      - RecAug:
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - KeepKeys:
          keep_keys: ["image", "label_ctc", "label_gtc", "length", "valid_ratio"]
  loader:
    shuffle: true
    batch_size_per_card: 64
    drop_last: true
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /
    label_file_list: ["/content/iam_labels/val_list.txt"]
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - KeepKeys:
          keep_keys: ["image", "label_ctc", "label_gtc", "length", "valid_ratio"]
  loader:
    shuffle: false
    batch_size_per_card: 64
    num_workers: 4
'''

with open("configs/rec/PP-OCRv4/en_PP-OCRv4_rec_iam.yml", "w") as f:
    f.write(config_yaml)

print("Config written.")


## 7. Train

Expect roughly 2-4 hours on a T4 for 30 epochs. Colab can disconnect on long idle sessions -- keep the tab open, and checkpoints save every 5 epochs to `/content/output/iam_finetune` so you can resume if it drops.

In [ ]:
!python tools/train.py -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec_iam.yml


## 8. Copy checkpoints to Drive periodically (run any time during/after training)

In [ ]:
!cp -r /content/output/iam_finetune "$DRIVE_OUTPUT_DIR/"
print("Checkpoints backed up to Drive.")


## 9. Export the best checkpoint for inference

In [ ]:
!python tools/export_model.py \
    -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec_iam.yml \
    -o Global.pretrained_model=/content/output/iam_finetune/best_accuracy \
       Global.save_inference_dir=/content/output/iam_finetune_infer

!zip -qr /content/iam_finetune_infer.zip /content/output/iam_finetune_infer
!cp /content/iam_finetune_infer.zip "$DRIVE_OUTPUT_DIR/"
print("Exported inference model saved to Drive as iam_finetune_infer.zip")


## 10. Next step

Download `iam_finetune_infer.zip` from Drive, unzip it next to your `CS-PADDLE-OCR.py` script as `iam_finetune_infer/`, then point PaddleOCR at it locally:

```python
ocr = PaddleOCR(
    ...,
    rec_model_dir="./iam_finetune_infer",
)
```

Then run `evaluate_ocr.py` (in `ocr_training/`) against your own labeled test images to check it's actually an improvement, not just a different output.